# VQA-RAD Ablation — chay bang CLI (`!python`)

`train.py` (huan luyen 1 run) · `eval.py` (danh gia 1 checkpoint) · `report.py` (gom bang). TACH RIENG.

⚠️ **Co kien truc khi eval PHAI trung khi train** (neu khong load_state_dict loi). Ten run = theo co.

Secret: khai bao `COMET_API_KEY`, `DISCORD_WEBHOOK_URL`, `HF_TOKEN` (Kaggle Secrets hoac `.env`). Chi tiet: `docs/plan/`.

In [ ]:
# Cell 1 - setup + pytest (verify src/ truoc khi chay)
import os, sys, subprocess
REPO = "/kaggle/input/medvqa-src"      # <-- sua duong dan repo (chua train.py, src/, tests/)
os.chdir(REPO)
# !pip install -q datasets tqdm huggingface_hub comet_ml python-dotenv
r = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], capture_output=True, text=True)
print(r.stdout[-2500:])
assert r.returncode == 0, "TESTS FAIL - dung lai"

## 0. SMOKE — chay nhanh kiem tra ca pipeline (~1-2 phut)

In [ ]:
!python train.py --run_name smoke_A1 --smoke --use_discord --use_comet
!python eval.py  --checkpoint runs/smoke_A1_best.pt --save_metrics runs/smoke_A1_metrics.json
!python report.py

## 1. NHOM A — 8 to hop attention (train)
Bo `--smoke` de chay that. Them `--use_discord --use_comet --use_hf_push --hf_repo_id user/repo` neu muon.

In [ ]:
!python train.py --run_name A1_000
!python train.py --run_name A2_001 --decoder_attention
!python train.py --run_name A3_010 --text_attention
!python train.py --run_name A4_011 --text_attention --decoder_attention
!python train.py --run_name A5_100 --image_attention
!python train.py --run_name A6_101 --image_attention --decoder_attention
!python train.py --run_name A7_110 --image_attention --text_attention
!python train.py --run_name A8_111 --image_attention --text_attention --decoder_attention

## 2. NHOM A — eval (co PHAI trung luc train). Them `--constrained_closed` cho ban rang buoc yes/no.

In [ ]:
!python eval.py --checkpoint runs/A1_000_best.pt --save_metrics runs/A1_000_metrics.json --save_predictions runs/A1_000_preds.csv
!python eval.py --checkpoint runs/A2_001_best.pt --decoder_attention --save_metrics runs/A2_001_metrics.json
!python eval.py --checkpoint runs/A3_010_best.pt --text_attention --save_metrics runs/A3_010_metrics.json
!python eval.py --checkpoint runs/A4_011_best.pt --text_attention --decoder_attention --save_metrics runs/A4_011_metrics.json
!python eval.py --checkpoint runs/A5_100_best.pt --image_attention --save_metrics runs/A5_100_metrics.json
!python eval.py --checkpoint runs/A6_101_best.pt --image_attention --decoder_attention --save_metrics runs/A6_101_metrics.json
!python eval.py --checkpoint runs/A7_110_best.pt --image_attention --text_attention --save_metrics runs/A7_110_metrics.json
!python eval.py --checkpoint runs/A8_111_best.pt --image_attention --text_attention --decoder_attention --save_metrics runs/A8_111_metrics.json
!python report.py     # xem run A nao tot nhat theo em_overall

## 3. NHOM B — Transformer text encoder
Thay `<A-FLAGS>` = co image/decoder cua run A tot nhat (KHONG gom text_attention). Vi du best = A8_111 -> `<A-FLAGS>` = `--image_attention --decoder_attention`.

In [ ]:
# Sua <A-FLAGS> theo best A (vd best=A8_111 -> --image_attention --decoder_attention)
!python train.py --run_name B1_trans_mean --text_encoder transformer --image_attention --decoder_attention
!python train.py --run_name B2_trans_attn --text_encoder transformer --text_attention --image_attention --decoder_attention
!python eval.py --checkpoint runs/B1_trans_mean_best.pt --text_encoder transformer --image_attention --decoder_attention --save_metrics runs/B1_trans_mean_metrics.json
!python eval.py --checkpoint runs/B2_trans_attn_best.pt --text_encoder transformer --text_attention --image_attention --decoder_attention --save_metrics runs/B2_trans_attn_metrics.json

## 4. NHOM C — ResNet-18 dong bang (train text + fusion). Vi du best A = A8_111.

In [ ]:
!python train.py --run_name C1_resnet_lstm --image_encoder resnet18_frozen --image_attention --text_attention --decoder_attention
!python train.py --run_name C2_resnet_best --image_encoder resnet18_frozen --text_encoder transformer --text_attention --image_attention --decoder_attention
!python eval.py --checkpoint runs/C1_resnet_lstm_best.pt --image_encoder resnet18_frozen --image_attention --text_attention --decoder_attention --save_metrics runs/C1_resnet_lstm_metrics.json
!python eval.py --checkpoint runs/C2_resnet_best_best.pt --image_encoder resnet18_frozen --text_encoder transformer --text_attention --image_attention --decoder_attention --save_metrics runs/C2_resnet_best_metrics.json

## 5. NHOM D — SFT -> RL (SCST)
`--load_checkpoint` = run tot nhat toan cuc; LAP LAI dung co kien truc cua run do. Vi du best = A8_111.

In [ ]:
!python train.py --run_name D1_scst --rl --load_checkpoint runs/A8_111_best.pt --image_attention --text_attention --decoder_attention
!python eval.py --checkpoint runs/D1_scst_best.pt --image_attention --text_attention --decoder_attention --save_metrics runs/D1_scst_metrics.json

## 6. Cham lai Gemma cu (dong bo) + tong hop bang

In [ ]:
# CSV Gemma 2 cot: answer_ref, answer_pred (xuat tu v3-vqa-*.ipynb)
!python eval.py --predictions_csv gemma_preds/gemma2b_test.csv --save_metrics runs/gemma-2b-lora_metrics.json
!python eval.py --predictions_csv gemma_preds/gemma4b_test.csv --save_metrics runs/gemma-4b-lora_metrics.json
!python report.py     # gom TAT CA runs/*_metrics.json -> runs/metrics_summary.csv